# Modul 4: RetailCo Procurement Analysis / Einkaufsanalyse

## DE
Dieses Notebook ergänzt das bestehende Finance-&-Controlling-Projekt um einen klar abgegrenzten, **synthetischen** Einkaufsdatensatz. Es zeigt Datenqualität, Einkaufs-KPIs, Lieferantenanalyse und Einkaufspreisabweichungen.

## EN
This notebook extends the existing Finance & Controlling project with a clearly separated **synthetic** procurement dataset. It demonstrates data quality checks, procurement KPIs, supplier analysis and purchase price variance (PPV).

**Important / Wichtig:** The procurement data are synthetic portfolio data. They are **not SAP data** and are not presented as real company purchasing transactions.


## 1. Business Questions / Geschäftsfragen

- Wie hoch ist das Einkaufsvolumen? / What is total purchase spend?
- Welche Lieferanten und Warengruppen konzentrieren das Einkaufsvolumen? / Which suppliers and categories concentrate spend?
- Wo liegen Einkaufspreisabweichungen? / Where do purchase price variances occur?
- Sind die Reporting-Daten vollständig und konsistent? / Are reporting data complete and consistent?


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..")
path = ROOT / "data" / "procurement" / "purchase_orders.csv"
po = pd.read_csv(path, parse_dates=["PO_Date"])
po.head()


,PO_ID,PO_Date,Supplier_ID,Supplier_Name,Category,Product,Quantity,Actual_Unit_Price,Planned_Unit_Price,Actual_Spend,Planned_Spend,PPV,PPV_Pct,Year,Month,Year_Month
0,PO100000,2015-01-05,SUP001,Supplier A,Accessories,Fender Set - Mountain,83,8.78,8.25,728.74,684.75,43.99,0.064242,2015,1,2015-01
1,PO100001,2015-01-05,SUP006,Supplier F,Accessories,ML Mountain Tire,213,11.57,11.94,2464.41,2543.22,-78.81,-0.030988,2015,1,2015-01
2,PO100002,2015-01-05,SUP008,Supplier H,Accessories,Touring Tire Tube,12,2.28,2.23,27.36,26.76,0.60,0.022422,2015,1,2015-01
3,PO100003,2015-01-06,SUP008,Supplier H,Accessories,"Sport-100 Helmet, Blue",265,14.39,14.18,3813.35,3757.70,55.65,0.014810,2015,1,2015-01
4,PO100004,2015-01-06,SUP004,Supplier D,Accessories,Fender Set - Mountain,110,8.07,8.54,887.70,939.40,-51.70,-0.055035,2015,1,2015-01


## 2. Data Quality / Datenqualität

Critical checks are performed before KPI calculation. / Kritische Prüfungen erfolgen vor der KPI-Berechnung.

In [2]:
checks = pd.DataFrame({
    "Check": [
        "Duplicate PO IDs",
        "Missing Supplier IDs",
        "Missing Unit Prices",
        "Invalid Quantities",
        "Invalid Dates",
        "Negative Spend",
    ],
    "Issue_Count": [
        po["PO_ID"].duplicated().sum(),
        po["Supplier_ID"].isna().sum(),
        po[["Actual_Unit_Price","Planned_Unit_Price"]].isna().any(axis=1).sum(),
        (po["Quantity"] <= 0).sum(),
        pd.to_datetime(po["PO_Date"], errors="coerce").isna().sum(),
        (po["Actual_Spend"] < 0).sum(),
    ]
})
checks["Status"] = np.where(checks["Issue_Count"] == 0, "PASS", "REVIEW")
checks


,Check,Issue_Count,Status
0,Duplicate PO IDs,0,PASS
1,Missing Supplier IDs,0,PASS
2,Missing Unit Prices,0,PASS
3,Invalid Quantities,0,PASS
4,Invalid Dates,0,PASS
5,Negative Spend,0,PASS


## 3. Core Procurement KPIs / Zentrale Einkaufs-KPIs

In [3]:
kpis = {
    "Purchase Spend / Einkaufsvolumen": po["Actual_Spend"].sum(),
    "Planned Spend / Geplantes Einkaufsvolumen": po["Planned_Spend"].sum(),
    "PPV / Einkaufspreisabweichung": po["PPV"].sum(),
    "PPV %": po["PPV"].sum() / po["Planned_Spend"].sum(),
    "Suppliers / Lieferanten": po["Supplier_ID"].nunique(),
    "Purchase Orders / Bestellungen": po["PO_ID"].nunique(),
    "Average PO Value / Durchschnittlicher Bestellwert": po["Actual_Spend"].mean(),
}
pd.Series(kpis)


Purchase Spend / Einkaufsvolumen                     4.824721e+06
Planned Spend / Geplantes Einkaufsvolumen            4.771530e+06
PPV / Einkaufspreisabweichung                        5.319069e+04
PPV %                                                1.114751e-02
Suppliers / Lieferanten                              1.200000e+01
Purchase Orders / Bestellungen                       9.000000e+02
Average PO Value / Durchschnittlicher Bestellwert    5.360801e+03
dtype: float64

## 4. Supplier Analysis / Lieferantenanalyse

In [4]:
supplier = (po.groupby(["Supplier_ID","Supplier_Name"], as_index=False)
             .agg(Purchase_Spend=("Actual_Spend","sum"),
                  Planned_Spend=("Planned_Spend","sum"),
                  PPV=("PPV","sum"),
                  Purchase_Orders=("PO_ID","nunique")))
supplier["PPV_Pct"] = supplier["PPV"] / supplier["Planned_Spend"]
supplier.sort_values("Purchase_Spend", ascending=False)


,Supplier_ID,Supplier_Name,Purchase_Spend,Planned_Spend,PPV,Purchase_Orders,PPV_Pct
7,SUP008,Supplier H,988042.42,961003.09,27039.33,61,0.028137
3,SUP004,Supplier D,930909.21,889721.49,41187.72,91,0.046293
9,SUP010,Supplier J,431501.33,424189.44,7311.89,44,0.017237
2,SUP003,Supplier C,371570.53,386503.63,-14933.10,106,-0.038636
1,SUP002,Supplier B,365157.68,371345.44,-6187.76,108,-0.016663
5,SUP006,Supplier F,338849.74,345243.30,-6393.56,93,-0.018519
0,SUP001,Supplier A,325844.50,322810.62,3033.88,112,0.009398
4,SUP005,Supplier E,288277.69,284894.78,3382.91,90,0.011874
11,SUP012,Supplier L,263141.41,257902.73,5238.68,28,0.020313
10,SUP011,Supplier K,206698.46,202498.43,4200.03,40,0.020741


## 5. Category Analysis / Warengruppenanalyse

In [5]:
category = (po.groupby("Category", as_index=False)
             .agg(Purchase_Spend=("Actual_Spend","sum"),
                  Planned_Spend=("Planned_Spend","sum"),
                  PPV=("PPV","sum"),
                  Purchase_Orders=("PO_ID","nunique")))
category["PPV_Pct"] = category["PPV"] / category["Planned_Spend"]
category.sort_values("Purchase_Spend", ascending=False)


,Category,Purchase_Spend,Planned_Spend,PPV,Purchase_Orders,PPV_Pct
1,Bikes,3338375.30,3291904.02,46471.28,22,0.014117
0,Accessories,767261.08,764364.64,2896.44,696,0.003789
2,Clothing,719084.54,715261.57,3822.97,182,0.005345


## 6. Monthly Trend / Monatlicher Trend

In [6]:
monthly = (po.assign(Year_Month=po["PO_Date"].dt.to_period("M").astype(str))
           .groupby("Year_Month", as_index=False)
           .agg(Purchase_Spend=("Actual_Spend","sum"),
                Planned_Spend=("Planned_Spend","sum"),
                PPV=("PPV","sum")))
monthly["PPV_Pct"] = monthly["PPV"] / monthly["Planned_Spend"]
monthly


,Year_Month,Purchase_Spend,Planned_Spend,PPV,PPV_Pct
0,2015-01,159490.55,163809.86,-4319.31,-0.026368
1,2015-02,120119.39,119891.34,228.05,0.001902
2,2015-03,581242.37,574674.89,6567.48,0.011428
3,2015-04,819352.61,793769.34,25583.27,0.032230
4,2015-05,665217.05,653073.43,12143.62,0.018595
5,2015-06,452444.44,466845.53,-14401.09,-0.030848
6,2015-07,270899.58,272947.17,-2047.59,-0.007502
7,2015-08,647150.40,637537.27,9613.13,0.015079
8,2015-09,145788.19,145849.29,-61.10,-0.000419
9,2015-10,489089.51,477950.92,11138.59,0.023305


## 7. Management Insights / Management-Erkenntnisse

Use the calculated tables above to write three factual insights. / Die folgenden Tabellen bilden die Grundlage für drei sachliche Management-Erkenntnisse.

1. Highest-spend category / Warengruppe mit höchstem Einkaufsvolumen
2. Supplier with highest spend / Lieferant mit höchstem Einkaufsvolumen
3. Category or supplier with largest absolute PPV / Größte absolute Einkaufspreisabweichung

**Interpretation rule / Interpretationsregel:** Positive PPV means actual purchase prices were above plan; negative PPV means actual purchase prices were below plan.


## 8. Limitations / Einschränkungen

- Procurement data are synthetic and created for portfolio demonstration.
- Planned prices are synthetic reference prices, not a historical management budget.
- The analysis demonstrates transferable procurement-controlling logic 
